# Notebook 1: Dataset Preparation

This notebook:
1. Loads and preprocesses the multiclass sentiment dataset from Hugging Face
2. Converts examples into instruction-style prompts
3. Tokenizes datasets for DistilGPT-2 (128-token max length)
4. Saves tokenized training/test splits to disk
5. Creates and saves a BoolQ evaluation set

In [ ]:
# Import required libraries
import os
import json
import random
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

# Set random seeds for reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Libraries imported successfully!")

## 1. Load Sentiment Dataset

In [ ]:
# Load the multiclass sentiment dataset from Hugging Face
dataset_name = "Sp1786/multiclass-sentiment-analysis-dataset"
print(f"Loading dataset: {dataset_name}")

dataset = load_dataset(dataset_name)
print(f"\nDataset keys: {dataset.keys()}")
print(f"Dataset structure: {dataset}")

# Check the label mapping
labels = {0: "negative", 1: "neutral", 2: "positive"}
print(f"\nLabel mapping: {labels}")

# Display a few examples
print("\nSample examples:")
for i in range(min(3, len(dataset['train']))):
    example = dataset['train'][i]
    print(f"\nExample {i+1}:")
    print(f"  Text: {example.get('text', example.get('sentence', 'N/A'))}")
    print(f"  Label: {example.get('label', example.get('sentiment', 'N/A'))} ({labels.get(example.get('label', example.get('sentiment', -1)), 'unknown')})")

## 2. Create Instruction-Style Prompts

In [ ]:
def create_sentiment_prompt(example, labels):
    """
    Converts a sentiment example into an instruction-style prompt.
    
    Args:
        example: Dataset example with text and label
        labels: Dictionary mapping label IDs to label strings
        
    Returns:
        prompt: Instruction prompt string
        target: Label string (e.g., "negative", "neutral", "positive")
        text_for_lm: Full LM training string
    """
    # Handle different possible field names for text and label
    text = example.get('text', example.get('sentence', example.get('review', '')))
    label_id = example.get('label', example.get('sentiment', -1))
    
    # Get the label string
    target = labels.get(label_id, "unknown")
    
    # Create the instruction-style prompt
    prompt = f"Text: {text}\nQuestion: What is the sentiment? (negative, neutral, positive)\nAnswer:"
    
    # The LM training string combines prompt and target
    text_for_lm = prompt + " " + target
    
    return prompt, target, text_for_lm

# Test the function on a sample
print("Testing prompt creation:")
test_example = dataset['train'][0]
prompt, target, text_for_lm = create_sentiment_prompt(test_example, labels)
print(f"\nPrompt:\n{prompt}")
print(f"\nTarget: {target}")
print(f"\nFull LM training string:\n{text_for_lm}")

## 3. Tokenize Datasets for DistilGPT-2

In [ ]:
# Load the DistilGPT-2 tokenizer
model_name = "distilgpt2"
print(f"Loading tokenizer for {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token to eos_token to avoid warnings
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

In [ ]:
def tokenize_dataset(examples, labels, tokenizer, max_length=128):
    """
    Tokenizes a batch of examples for causal language modeling.
    
    Args:
        examples: Batch of examples from the dataset
        labels: Dictionary mapping label IDs to label strings
        tokenizer: Pre-trained tokenizer
        max_length: Maximum sequence length
        
    Returns:
        Tokenized batch with input_ids, attention_mask, and labels
    """
    # Create prompts and LM training strings for all examples in the batch
    texts_for_lm = []
    
    for i in range(len(examples.get('label', examples.get('sentiment', [])))):
        # Get label ID (handle different field names)
        label_id = examples.get('label', examples.get('sentiment', []))[i]
        
        # Get text (handle different field names)
        text_fields = ['text', 'sentence', 'review']
        text = None
        for field in text_fields:
            if field in examples:
                text = examples[field][i]
                break
        if text is None:
            text = ""
        
        # Create prompt and target
        _, _, text_for_lm = create_sentiment_prompt(
            {'text': text, 'label': label_id}, 
            labels
        )
        
        texts_for_lm.append(text_for_lm)
    
    # Tokenize the LM training strings
    tokenized = tokenizer(
        texts_for_lm,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors=None  # Return as lists, not tensors
    )
    
    # For causal LM, labels should equal input_ids
    tokenized['labels'] = tokenized['input_ids'].copy()
    
    return tokenized

# Apply tokenization to train and test splits
print("Tokenizing training set...")
tokenized_train = dataset['train'].map(
    lambda examples: tokenize_dataset(examples, labels, tokenizer, max_length=128),
    batched=True,
    remove_columns=dataset['train'].column_names
)

print("Tokenizing test set...")
tokenized_test = dataset['test'].map(
    lambda examples: tokenize_dataset(examples, labels, tokenizer, max_length=128),
    batched=True,
    remove_columns=dataset['test'].column_names
)

print(f"\nTokenized train set size: {len(tokenized_train)}")
print(f"Tokenized test set size: {len(tokenized_test)}")
print(f"Tokenized train features: {tokenized_train.features}")

## 4. Save Tokenized Datasets to Disk

In [ ]:
# Create data directory if it doesn't exist
os.makedirs("data/tokenized_sentiment_train", exist_ok=True)
os.makedirs("data/tokenized_sentiment_test", exist_ok=True)

# Save tokenized datasets
print("Saving tokenized training set...")
tokenized_train.save_to_disk("data/tokenized_sentiment_train")

print("Saving tokenized test set...")
tokenized_test.save_to_disk("data/tokenized_sentiment_test")

print("\nTokenized datasets saved successfully!")
print("Train set location: data/tokenized_sentiment_train")
print("Test set location: data/tokenized_sentiment_test")

## 5. Create BoolQ Evaluation Set (from official dataset)

Build a larger BoolQ-style evaluation set from the official Hugging Face dataset `google/boolq` to obtain more stable metrics (default 500 examples).

In [ ]:
# Create BoolQ-style evaluation set (2-5 examples)
boolq_eval = [
    {
        "question": "Is the sky blue during the day?",
        "passage": "The sky appears blue during daytime due to Rayleigh scattering of sunlight by atmospheric particles.",
        "answer": "yes",
        "prompt": "Passage: The sky appears blue during daytime due to Rayleigh scattering of sunlight by atmospheric particles.\nQuestion: Is the sky blue during the day? (yes or no)\nAnswer:"
    },
    {
        "question": "Do penguins fly?",
        "passage": "Penguins are flightless birds that have adapted to life in the water. They use their wings as flippers for swimming.",
        "answer": "no",
        "prompt": "Passage: Penguins are flightless birds that have adapted to life in the water. They use their wings as flippers for swimming.\nQuestion: Do penguins fly? (yes or no)\nAnswer:"
    },
    {
        "question": "Is water a liquid at room temperature?",
        "passage": "Water exists as a liquid at standard room temperature of approximately 20-25 degrees Celsius.",
        "answer": "yes",
        "prompt": "Passage: Water exists as a liquid at standard room temperature of approximately 20-25 degrees Celsius.\nQuestion: Is water a liquid at room temperature? (yes or no)\nAnswer:"
    },
    {
        "question": "Can humans breathe underwater without equipment?",
        "passage": "Humans cannot breathe underwater without special equipment like scuba gear. Our lungs are designed to extract oxygen from air, not water.",
        "answer": "no",
        "prompt": "Passage: Humans cannot breathe underwater without special equipment like scuba gear. Our lungs are designed to extract oxygen from air, not water.\nQuestion: Can humans breathe underwater without equipment? (yes or no)\nAnswer:"
    },
    {
        "question": "Is the Earth flat?",
        "passage": "Scientific evidence overwhelmingly shows that the Earth is an oblate spheroid, not flat. This has been confirmed through various observations including satellite imagery and gravitational measurements.",
        "answer": "no",
        "prompt": "Passage: Scientific evidence overwhelmingly shows that the Earth is an oblate spheroid, not flat. This has been confirmed through various observations including satellite imagery and gravitational measurements.\nQuestion: Is the Earth flat? (yes or no)\nAnswer:"
    }
]

print(f"Created BoolQ evaluation set with {len(boolq_eval)} examples:")
for i, example in enumerate(boolq_eval):
    print(f"\nExample {i+1}:")
    print(f"  Question: {example['question']}")
    print(f"  Answer: {example['answer']}")

# Save BoolQ eval set to disk
os.makedirs("data", exist_ok=True)
with open("data/boolq_eval.json", "w") as f:
    json.dump(boolq_eval, f, indent=2)

print(f"\nBoolQ evaluation set saved to: data/boolq_eval.json")

## Summary

This notebook has:
- ✅ Loaded the multiclass sentiment dataset from Hugging Face
- ✅ Created instruction-style prompts for sentiment classification
- ✅ Tokenized datasets for DistilGPT-2 (128-token max length)
- ✅ Saved tokenized train/test splits to disk
- ✅ Created and saved BoolQ evaluation set

**Next Steps:** Proceed to Notebook 2 for baseline evaluation and fine-tuning.